In [1]:
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
import pickle
import os

d:\Projects\Sentiment_analsis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kazanova/sentiment140")

print("Path to dataset files:", path)


Path to dataset files: C:\Users\PC\.cache\kagglehub\datasets\kazanova\sentiment140\versions\2


In [3]:
os.listdir(path=path)

['training.1600000.processed.noemoticon.csv']

In [4]:
df=pd.read_csv(path+"/training.1600000.processed.noemoticon.csv",encoding='latin-1',header=None)
df=df[[0,5]]
df.columns = ['polarity', 'text']
df.head()

,polarity,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 2 columns):
 #   Column    Non-Null Count    Dtype
---  ------    --------------    -----
 0   polarity  1600000 non-null  int64
 1   text      1600000 non-null  str  
dtypes: int64(1), str(1)
memory usage: 24.4 MB


In [6]:
def clean_text(text):
    return text.lower()

df['clean_text']=df['text'].apply(clean_text)
df.head()

,polarity,text,clean_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...","@switchfoot http://twitpic.com/2y1zl - awww, t..."
1,0,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...,@kenichan i dived many times for the ball. man...
3,0,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all....","@nationwideclass no, it's not behaving at all...."


In [7]:
df=df[df['polarity']!=2]

In [8]:
X=df['clean_text']
y=df['polarity']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

vectorizer=TfidfVectorizer(max_features=50000,ngram_range=(1,2)) 
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

logreg=LogisticRegression(max_iter=1500)
logreg.fit(X_train_vectorized,y_train)
pred=logreg.predict(X_test_vectorized)

print(f"the accuracy of the logreg is{accuracy_score(pred,y_test)} ")
print(f"the calassificatio acc of the logreg is{classification_report(pred,y_test)} ")

svc=LinearSVC()
svc.fit(X_train_vectorized,y_train)
pred_svc=svc.predict(X_test_vectorized)

print(f"the accuracy of the svc is{accuracy_score(pred_svc,y_test)} ")
print(f"the calassificatio acc of the svc is{classification_report(pred_svc,y_test)} ")


with open("model.pkl", "wb") as f:
    pickle.dump(svc, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

the accuracy of the logreg is0.81874375 
the calassificatio acc of the logreg is              precision    recall  f1-score   support

           0       0.81      0.83      0.82    156064
           4       0.83      0.81      0.82    163936

    accuracy                           0.82    320000
   macro avg       0.82      0.82      0.82    320000
weighted avg       0.82      0.82      0.82    320000
 
the accuracy of the svc is0.81735 
the calassificatio acc of the svc is              precision    recall  f1-score   support

           0       0.80      0.82      0.81    155526
           4       0.83      0.81      0.82    164474

    accuracy                           0.82    320000
   macro avg       0.82      0.82      0.82    320000
weighted avg       0.82      0.82      0.82    320000
 
